# HadISD Data Download Notebook

This notebook will help you download a subset of the HadISD dataset directly from the Met Office website. The data will be stored in a user-specified directory (or a sensible default), and extracted for further processing (e.g., conversion to Zarr).

- **Source:** [HadISD v3.4.0.2023f](https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html)
- **Instructions:**
    1. Set the download directory (or use the default).
    2. Download the data using Python's `requests` package.
    3. Extract the `.tar.gz` archive.
    4. The extracted files will be ready for use in the next notebook (`HadISD_to_zarr.ipynb`).

> **Note:** Download size is large. Ensure you have sufficient disk space and a stable internet connection.

In [1]:
import requests
from tqdm.auto import tqdm
import tarfile
import gzip
import shutil

### Retrieve Path to Download Directory
The download location will default to a folder named "HadISD_data" in your home directory.<br>
If you want to change this, you can do so in the `Data_config.ipynb` configuration notebook. <br>


In [ ]:
%run Data_Config.ipynb
print(f"Data will be downloaded to: {download_dir}")   

### Download HadISD Data
The following code will download the HadISD data files. Some files take longer to download than others depending on time of day. To download different WMO datasets, you can change `wmo_id_range` in the `Data_Config.ipynb` notebook .

The full list of available data can be found here:
https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [ ]:
# Explain why stations are split into ranges, file size, and how it's not neccesssary to download all stations. 

In [ ]:
print(f"Downloading HadISD data for WMO range: {wmo_id_range}")

In [ ]:
wmo_id_range = wmo_id_range # This has been defined in HadISD_data_config.ipynb

wmo_str = f"WMO_{wmo_id_range}"
url = f"https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/{wmo_str}.tar.gz"
tar_name = f"{wmo_str}.tar"
filename = download_dir / tar_name

In [ ]:
# Get remote file size using HTTP HEAD
head = requests.head(url, allow_redirects=True)
remote_size = int(head.headers.get('content-length', 0))

local_size = filename.stat().st_size if filename.exists() else 0

if filename.exists() and local_size == remote_size:
    print(f"File already fully downloaded: {filename} ({local_size/1024**2:.2f} MB)")
else:
    headers = {}
    mode = 'wb'
    initial_pos = 0
    if filename.exists() and local_size < remote_size:
        headers['Range'] = f'bytes={local_size}-'
        mode = 'ab'
        initial_pos = local_size
        print(f"Resuming download for {filename.name} at {local_size/1024**2:.2f} MB...")
    else:
        print(f"Starting download for {filename.name}...")

    response = requests.get(url, stream=True, headers=headers)
    total = remote_size

    with open(filename, mode) as f, tqdm(
        desc=f"Downloading {filename.name}",
        total=total,
        initial=initial_pos,
        unit='B', unit_scale=True, unit_divisor=1024
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    final_size = filename.stat().st_size
    if final_size == remote_size:
        print(f"Download complete: {filename} ({final_size/1024**2:.2f} MB)")
    else:
        print(f"Warning: Download incomplete. Local size: {final_size}, Remote size: {remote_size}")

# Possibly also add check to see if netcdf files esist for the downloaded tar file, if so then don't download again

### Extract Tar Files and Move to Netcdf Subfolder

In [ ]:
extract_dir = download_dir / tar_name.replace('.tar', '')
extract_dir.mkdir(exist_ok=True)

extracted_files = list(extract_dir.glob('*'))
if extracted_files:
    print(f"Extraction directory '{extract_dir}' already contains {len(extracted_files)} files. Skipping extraction.")
elif filename.exists():
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall(path=extract_dir)
    extracted_files = list(extract_dir.glob('*'))
    if extracted_files:
        print(f"Extraction successful. {len(extracted_files)} files found in {extract_dir}.")
        # Delete the tar file after extraction
        filename.unlink()
        print(f"Deleted tar file: {filename}")
    else:
        print(f"Warning: No files extracted to {extract_dir}. Tar file will not be deleted.")
        raise RuntimeError("Extraction failed, tar file not deleted.")
else:
    print(f"No tar file found and extraction directory is empty. Nothing to extract.")
    raise FileNotFoundError(f"Missing tar file: {filename}")


In [ ]:
# Create subfolder for netcdf
netcdf_dir = download_dir / "netcdf"
netcdf_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Move extracted .nc files into netcdf_dir after extraction
num_files = 0
for gz_path in extract_dir.glob('*.nc.gz'):
    nc_path = gz_path.with_suffix('')  # Remove .gz extension
    with gzip.open(gz_path, 'rb') as f_in, open(nc_path, 'wb') as f_out:
        f_out.write(f_in.read())
    gz_path.unlink()  # Delete the .gz file after extraction
    shutil.move(str(nc_path), netcdf_dir / nc_path.name)
    num_files += 1

print(f"{num_files} .nc files have been extracted, cleaned up, and moved to the netcdf directory: {netcdf_dir}")

# Delete the extraction directory after processing
try:
    shutil.rmtree(extract_dir)
    print(f"Deleted extraction directory: {extract_dir}")
except Exception as e:
    print(f"Could not delete extraction directory {extract_dir}: {e}")